Base parcial com colunas selecionadas

In [9]:
from pathlib import Path
import pandas as pd
import json

base_path = Path.cwd().parent
file_path = base_path / "data" / "processed" / "Base_de_Dados_Processada.csv"
features_path = base_path / "data" / "processed" / "features_perfil.json"

# Colunas alvo
target_col_ideb_af = 'Normalizado (0-100): SDG4_15_IDEB_AF'
target_col_ideb_ai = 'Normalizado (0-100): SDG4_16_IDEB_AI'

with open(features_path, "r", encoding="utf-8") as arquivo:
    features = json.load(arquivo)

partial_base_cols = [
    target_col_ideb_af,  
    target_col_ideb_ai   
]

partial_base_cols.extend(features)

df_partial = pd.read_csv(
    file_path, 
    usecols=partial_base_cols,
    sep=','  
)

targets = df_partial[[target_col_ideb_af, target_col_ideb_ai]].copy()

Importação de modelos e dependências 

In [10]:

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Modelos
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

Separação das variáveis alvo 

In [11]:
target_col_ideb_af = 'Normalizado (0-100): SDG4_15_IDEB_AF'
target_col_ideb_ai = 'Normalizado (0-100): SDG4_16_IDEB_AI'

targets = df_partial[[target_col_ideb_af, target_col_ideb_ai]].copy()

Divisão da base de dados para treino e teste e execução dos modelos 

In [12]:

def avaliar_modelos_por_alvo(df_base, coluna_alvo, colunas_remover):
    print(f"\n--- Avaliação Baseline: {coluna_alvo} ---")
    
    df_clean = df_base.dropna(subset=[coluna_alvo])
    
    X = df_clean.drop(columns=colunas_remover)
    y = df_clean[coluna_alvo].values.ravel()
    
    # Divisão em Treino e Teste (80/20)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Padronização das features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    modelos = {
        'Random Forest': RandomForestRegressor(random_state=42),
        'XGBoost': XGBRegressor(random_state=42),
        'LightGBM': LGBMRegressor(random_state=42, verbose=-1),
        'CatBoost': CatBoostRegressor(random_state=42, verbose=0)
    }

    resultados = []
    for nome, modelo in modelos.items():
        modelo.fit(X_train_scaled, y_train)
        y_pred = modelo.predict(X_test_scaled)
        
        resultados.append({
            'Modelo': nome,
            'MAE': mean_absolute_error(y_test, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
            'R²': r2_score(y_test, y_pred)
        })
        
    return pd.DataFrame(resultados).sort_values(by='R²', ascending=False)

Exibição dos resultados 

In [13]:

# Anos Finais
resultados_af = avaliar_modelos_por_alvo(df_partial, target_col_ideb_af, targets)
display(resultados_af)

# Anos Iniciais
resultados_ai = avaliar_modelos_por_alvo(df_partial, target_col_ideb_ai, targets)
display(resultados_ai)


--- Avaliação Baseline: Normalizado (0-100): SDG4_15_IDEB_AF ---


,Modelo,MAE,RMSE,R²
0,Random Forest,9.490126,12.883886,0.156769
3,CatBoost,9.586347,12.917838,0.152319
2,LightGBM,9.573971,13.020790,0.138754
1,XGBoost,10.346435,13.835045,0.027670



--- Avaliação Baseline: Normalizado (0-100): SDG4_16_IDEB_AI ---


,Modelo,MAE,RMSE,R²
3,CatBoost,10.684485,13.946894,0.395444
2,LightGBM,10.661353,13.995574,0.391217
0,Random Forest,10.660006,14.050071,0.386466
1,XGBoost,11.640087,15.064793,0.294645
